In [ ]:
%cd ..

In [1]:
# unidecode
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [2]:
import requests
import pandas as pd
import re
import os

# ===== CONFIG =====
DOMAIN = os.getenv("NOCODB_URL")
TABLE_ID = os.getenv("NOCODB_TABLE_ID")
TOKEN = os.getenv("NOCODB_TOKEN")
PAGE_SIZE = 1000

In [3]:


def get_nocodb_data():
    url = f"{DOMAIN}api/v2/tables/{TABLE_ID}/records"
    headers = {
        "Accept": "application/json",
        "xc-token": TOKEN
    }
    
    all_records = []
    offset = 0
    
    while True:
        params = {
            "limit": PAGE_SIZE,
            "offset": offset
        }
        
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() # Kiểm tra lỗi HTTP
        
        data = response.json().get('list', [])
        
        if not data:
            break
            
        all_records.extend(data)
        offset += PAGE_SIZE
        
        # Nếu số lượng bản ghi lấy về ít hơn PAGE_SIZE nghĩa là đã hết dữ liệu
        if len(data) < PAGE_SIZE:
            break

    # Tạo DataFrame
    df = pd.DataFrame(all_records)
    
    if df.empty:
        return df

    # ===== EXPAND COLUMNS (Xử lý các cột lồng nhau) =====
    # Tương đương Table.ExpandRecordColumn trong Power BI
    if 'dws_kpi_criterias' in df.columns:
        df['dws_kpi_criterias.Name'] = df['dws_kpi_criterias'].apply(lambda x: x.get('Name') if isinstance(x, dict) else None)
        
    if 'dws_kpi_systems' in df.columns:
        df['dws_kpi_systems.Name'] = df['dws_kpi_systems'].apply(lambda x: x.get('Name') if isinstance(x, dict) else None)

    # ===== CLEANING (Làm sạch dữ liệu) =====
    def clean_system_name(text):
        if not isinstance(text, str):
            return text
        
        # 1. Bỏ phần trong ngoặc [ ] và cả dấu ngoặc
        # Regex này tìm nội dung từ dấu [ đến hết
        cleaned = re.sub(r'\[.*\]', '', text).strip()
        
        # 2. Thay thế "Tỉ lệ" thành "Tỷ lệ"
        cleaned = cleaned.replace("Tỉ lệ", "Tỷ lệ")
        
        return cleaned

    if 'dws_kpi_systems.Name' in df.columns:
        df['dws_kpi_systems.Name'] = df['dws_kpi_systems.Name'].apply(clean_system_name)

    return df


import pandas as pd
import re
import unidecode

def clean_column_names(df):
    def transform(column_name):
        column_name = column_name.lower()
        column_name = unidecode.unidecode(column_name)
        column_name = re.sub(r'[^a-z0-9]', '_', column_name)
        column_name = re.sub(r'_+', '_', column_name)
        column_name = column_name.strip('_')
        return column_name

    df.columns = [transform(col) for col in df.columns]
    return df


In [4]:
# Chạy và hiển thị kết quả
df_final = get_nocodb_data()

In [5]:
df_final = clean_column_names(df_final)

In [6]:
print(df_final.head(1))

     id title                  createdat                  updatedat  \
0  1956  None  2025-08-14 12:02:27+00:00  2025-09-25 10:28:32+00:00   

    starttime     endtime  value  before1periodvalue  before2periodsvalue  \
0  2024-12-29  2025-01-04    0.0                 0.0                  0.0   

   targetdifference  ...              calculatetime  error  isempty  \
0             -98.0  ...  2025-08-18 04:01:44+00:00   None     True   

              dws_kpi_systems  customername customertype  \
0  {'Id': 9, 'Name': 'Masan'}         Masan         VVIP   

                                 dws_kpi_criterias productname  \
0  {'Id': 33, 'Name': 'Tỉ lệ agent online có log'}        SIEM   

      dws_kpi_criterias_name  dws_kpi_systems_name  
0  Tỉ lệ agent online có log                 Masan  

[1 rows x 33 columns]


In [7]:
records = df_final.to_dict(orient="records")

In [8]:
records[0]

{'id': 1956,
 'title': None,
 'createdat': '2025-08-14 12:02:27+00:00',
 'updatedat': '2025-09-25 10:28:32+00:00',
 'starttime': '2024-12-29',
 'endtime': '2025-01-04',
 'value': 0.0,
 'before1periodvalue': 0.0,
 'before2periodsvalue': 0.0,
 'targetdifference': -98.0,
 'before2periodsdifference': 0.0,
 'before1perioddifference': 0.0,
 'result': False,
 'trend': 'Maintain',
 'criteriatarget': 99.8,
 'criteriaperiodtype': 'Week',
 'criteriaoperator': '>=',
 'criteriaunit': '%',
 'dws_kpi_systems_id': 9.0,
 'dws_kpi_criterias_id': 33.0,
 'criteriacode': 'kpi_vcs_siem_agent_online_have_log_succ',
 'influxdbquery': 'SELECT last("value") FROM "kpi"."autogen"."kpi_daily" WHERE  (  "project" = \'siem\'  AND "customer" = \'masan\' AND "kpi_code" = \'kpi_vcs_siem_agent_online_have_log_succ\'  AND "environment" = \'product\')  AND time >= 1735430400000ms and time <= 1736035199999ms ORDER BY time ASC LIMIT 1',
 'influxdbresult': '{"results":[{"statement_id":0}]}',
 'calculatetime': '2025-08-18 04:

In [9]:

from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(records):
    resource_name= "kpi_systems"
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "/opt/datasets/crawlers/vcs/nocodb/data"
    ENABLE_STATE =False
    HIVE_DB = "nocodb_raw"
    crawl_mode = "static"
    schema_local_path = None

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    # =========================
    # Save new state
    # =========================
    if ENABLE_STATE and "updated_at" in df.columns:
        max_ts = get_max_updated_at_str(df)
        print("last state ", resource_name, "max_ts=", max_ts)
        max_ts = subtract_minutes(max_ts, 30)
        write_last_state(max_ts, resource_name)
        print("last state ", resource_name, "max_ts=", max_ts)

    elapsed = time.time() - start_time
    print("Loop {} took {:.3f}s".format(resource_name, elapsed))
    if len(records) < 100:
        print("No new data")
        out_of_data = True
        return True
    return False



In [10]:
fetch_resource_name_freshwork(records)

Start crawl :  kpi_systems
kpi_systems
Replace Upload  /opt/datasets/crawlers/vcs/nocodb/data/kpi_systems ./tmp/data/kpi_systems/data_kpi_systems_20260202_155203.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/nocodb/data/kpi_systems
Uploaded SQL definition
Loop kpi_systems took 45.676s


False